# HW1 solution

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset


def resolve_hw1_dir() -> Path:
    candidates = [Path.cwd(), Path.cwd() / "HW1"]
    for candidate in candidates:
        if (candidate / "IDS.py").exists() and (candidate / "simple_rules.py").exists() and (candidate / "NSL-KDD").exists():
            return candidate
    raise FileNotFoundError("Could not locate HW1 directory containing IDS.py, simple_rules.py, and NSL-KDD.")


HW1_DIR = resolve_hw1_dir()
if str(HW1_DIR) not in sys.path:
    sys.path.insert(0, str(HW1_DIR))

from IDS import IntrusionDetector
from simple_rules import AttackPlausibilityChecker


def fail(message: str) -> None:
    print(f"ERROR: {message}")
    raise RuntimeError(message)


def set_seed(seed: int = 42) -> None:
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_feature_names(features_json_path: Path) -> list[str]:
    with open(features_json_path, "r", encoding="utf-8") as f:
        feature_meta = json.load(f)
    return [item["name"] for item in feature_meta]


def preprocess_nslkdd(data_dir: Path, features_json_path: Path):
    feature_names = load_feature_names(features_json_path)
    categorical_cols = ["protocol_type", "service", "flag"]

    train_path = data_dir / "KDDTrain+.txt"
    test_path = data_dir / "KDDTest+.txt"

    train_df = pd.read_csv(train_path, header=None, names=feature_names)
    test_df = pd.read_csv(test_path, header=None, names=feature_names)

    y_train_multiclass = train_df["label"].astype(str).str.strip().str.rstrip(".")
    y_test_multiclass = test_df["label"].astype(str).str.strip().str.rstrip(".")

    X_train_raw = train_df.drop(columns=["label", "difficulty"]).copy()
    X_test_raw = test_df.drop(columns=["label", "difficulty"]).copy()

    X_all = pd.concat([X_train_raw, X_test_raw], axis=0, ignore_index=True)
    X_all_encoded = pd.get_dummies(X_all, columns=categorical_cols)
    X_all_encoded = X_all_encoded.reindex(sorted(X_all_encoded.columns), axis=1)

    n_train = len(X_train_raw)
    X_train_encoded = X_all_encoded.iloc[:n_train].copy()
    X_test_encoded = X_all_encoded.iloc[n_train:].copy()

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_encoded.values)
    X_test_scaled = scaler.transform(X_test_encoded.values)

    y_train_bin = (y_train_multiclass != "normal").astype(np.float32).to_numpy()
    y_test_bin = (y_test_multiclass != "normal").astype(np.float32).to_numpy()

    feature_names_after_encoding = X_train_encoded.columns.tolist()

    return (
        X_train_scaled,
        X_test_scaled,
        y_train_bin,
        y_test_bin,
        y_train_multiclass.to_numpy(),
        y_test_multiclass.to_numpy(),
        scaler,
        feature_names_after_encoding,
    )


def load_feature_metadata(features_json_path: Path) -> list[dict]:
    with open(features_json_path, "r", encoding="utf-8") as f:
        return json.load(f)


def build_modifiability_masks(feature_names_after_encoding: list[str], features_json_path: Path):
    feature_meta = load_feature_metadata(features_json_path)
    modifiability_by_name = {item["name"]: item["modifiability"] for item in feature_meta}

    high_mask = np.zeros(len(feature_names_after_encoding), dtype=bool)
    high_or_partial_mask = np.zeros(len(feature_names_after_encoding), dtype=bool)

    for i, encoded_name in enumerate(feature_names_after_encoding):
        base_name = encoded_name
        if "_" in encoded_name and encoded_name.split("_", 1)[0] in {"protocol", "service", "flag"}:
            if encoded_name.startswith("protocol_type_"):
                base_name = "protocol_type"
            elif encoded_name.startswith("service_"):
                base_name = "service"
            elif encoded_name.startswith("flag_"):
                base_name = "flag"

        level = modifiability_by_name.get(base_name, "NON_MODIFIABLE")
        if level == "HIGHLY_MODIFIABLE":
            high_mask[i] = True
            high_or_partial_mask[i] = True
        elif level == "PARTIALLY_MODIFIABLE":
            high_or_partial_mask[i] = True

    return high_mask, high_or_partial_mask


def map_attack_label_to_family(label: str) -> str:
    dos = {
        "back",
        "land",
        "neptune",
        "pod",
        "smurf",
        "teardrop",
        "apache2",
        "mailbomb",
        "processtable",
        "udpstorm",
    }
    probe = {"ipsweep", "nmap", "portsweep", "satan", "mscan", "saint"}
    r2l = {
        "ftp_write",
        "guess_passwd",
        "imap",
        "multihop",
        "phf",
        "spy",
        "warezclient",
        "warezmaster",
        "snmpgetattack",
        "snmpguess",
        "httptunnel",
        "sendmail",
        "named",
        "xlock",
        "xsnoop",
        "worm",
    }
    u2r = {"buffer_overflow", "loadmodule", "perl", "rootkit", "ps", "sqlattack", "xterm"}

    if label in dos:
        return "DoS"
    if label in probe:
        return "Probe"
    if label in r2l:
        return "R2L"
    if label in u2r:
        return "U2R"
    return "R2L"


def compute_binary_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    y_true = y_true.astype(np.int64)
    y_pred = y_pred.astype(np.int64)

    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())

    accuracy = (tp + tn) / max(1, tp + tn + fp + fn)
    precision = tp / max(1, tp + fp)
    recall = tp / max(1, tp + fn)
    f1 = 2 * precision * recall / max(1e-12, precision + recall)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


def train_ids(
    X_train: np.ndarray,
    y_train: np.ndarray,
    input_dim: int,
    batch_size: int = 128,
    lr: float = 0.001,
    epochs: int = 10,
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = IntrusionDetector(input_dim=input_dim).to(device)

    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    X_tensor = torch.tensor(X_train, dtype=torch.float32)
    y_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
    train_loader = DataLoader(TensorDataset(X_tensor, y_tensor), batch_size=batch_size, shuffle=True)

    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()
            preds = model(xb)
            loss = criterion(preds, yb)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * xb.size(0)

        epoch_loss = running_loss / len(train_loader.dataset)
        print(f"Epoch {epoch + 1:02d}/{epochs} | train_loss = {epoch_loss:.4f}")

    return model, device


def predict_binary(model: nn.Module, device: torch.device, X: np.ndarray) -> np.ndarray:
    model.eval()
    X_tensor = torch.tensor(X, dtype=torch.float32).to(device)
    with torch.no_grad():
        probs = model(X_tensor).squeeze(1).cpu().numpy()
    return (probs >= 0.5).astype(np.int64)

### Task 0: Data Preprocessing

In [2]:
set_seed(42)

base_dir = HW1_DIR
data_dir = base_dir / "NSL-KDD"
features_json_path = base_dir / "nslkdd_features.json"

(
    X_train,
    X_test,
    y_train_bin,
    y_test_bin,
    y_train_multiclass,
    y_test_multiclass,
    scaler,
    feature_names_after_encoding,
) = preprocess_nslkdd(data_dir, features_json_path)

if X_train.shape[0] == 0 or X_test.shape[0] == 0:
    fail("Empty dataset after preprocessing.")
if X_train.shape[1] != X_test.shape[1]:
    fail("Train/test feature dimension mismatch after preprocessing.")

preprocess_table = pd.DataFrame(
    [
        ["Train samples", X_train.shape[0]],
        ["Test samples", X_test.shape[0]],
        ["Feature count after one-hot", X_train.shape[1]],
    ],
    columns=["Item", "Value"],
)
print("\nTask 0: Preprocessing summary")
print(preprocess_table.to_string(index=False))


Task 0: Preprocessing summary
                       Item  Value
              Train samples 125973
               Test samples  22544
Feature count after one-hot    122


### Task 1: Train and Evaluate the IDS Binary Classifier

In [3]:
print("\nTraining..")
model, device = train_ids(
    X_train=X_train,
    y_train=y_train_bin,
    input_dim=X_train.shape[1],
    batch_size=128,
    lr=0.001,
    epochs=10,
)

y_pred_test = predict_binary(model, device, X_test)
metrics_all = compute_binary_metrics(y_test_bin, y_pred_test)

train_attack_labels = set(y_train_multiclass[y_train_multiclass != "normal"])
test_attack_mask = y_test_bin == 1
zero_day_mask = np.array(
    [(label not in train_attack_labels) and (label != "normal") for label in y_test_multiclass],
    dtype=bool,
)
non_zero_day_mask = test_attack_mask & (~zero_day_mask)

zero_day_acc = float(np.mean(y_pred_test[zero_day_mask] == y_test_bin[zero_day_mask])) if zero_day_mask.any() else float("nan")
non_zero_day_acc = (
    float(np.mean(y_pred_test[non_zero_day_mask] == y_test_bin[non_zero_day_mask]))
    if non_zero_day_mask.any()
    else float("nan")
)

full_metrics_table = pd.DataFrame(
    [
        ["Accuracy", metrics_all["accuracy"]],
        ["Precision", metrics_all["precision"]],
        ["Recall", metrics_all["recall"]],
        ["F1-score", metrics_all["f1"]],
    ],
    columns=["Metric", "Value"],
)
print("\nTask 1: Full-test metrics")
print(full_metrics_table.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

subset_table = pd.DataFrame(
    [
        ["Zero-day attacks", int(zero_day_mask.sum()), zero_day_acc],
        ["Non-zero-day attacks", int(non_zero_day_mask.sum()), non_zero_day_acc],
    ],
    columns=["Subset", "Sample Count", "Accuracy"],
)
print("\nTask 1: Subset accuracies")
print(subset_table.to_string(index=False, float_format=lambda x: f"{x:.4f}"))


Training..
Epoch 01/10 | train_loss = 0.0815
Epoch 02/10 | train_loss = 0.0308
Epoch 03/10 | train_loss = 0.0261
Epoch 04/10 | train_loss = 0.0251
Epoch 05/10 | train_loss = 0.0217
Epoch 06/10 | train_loss = 0.0201
Epoch 07/10 | train_loss = 0.0193
Epoch 08/10 | train_loss = 0.0189
Epoch 09/10 | train_loss = 0.0177
Epoch 10/10 | train_loss = 0.0180

Task 1: Full-test metrics
   Metric  Value
 Accuracy 0.7943
Precision 0.9288
   Recall 0.6917
 F1-score 0.7929

Task 1: Subset accuracies
              Subset  Sample Count  Accuracy
    Zero-day attacks          3750    0.5024
Non-zero-day attacks          9083    0.7698


### Task 2: Evasion with Constrained PGD

In [4]:
def pgd_evasion_single(
    model: nn.Module,
    device: torch.device,
    x_orig_np: np.ndarray,
    eps: float,
    alpha: float,
    n_iters: int,
    feature_mask_np: np.ndarray,
    train_min_np: np.ndarray,
    train_max_np: np.ndarray,
) -> np.ndarray:
    criterion = nn.BCELoss()

    x_orig = torch.tensor(x_orig_np, dtype=torch.float32, device=device).unsqueeze(0)
    delta = torch.zeros_like(x_orig)

    feature_mask = torch.tensor(feature_mask_np.astype(np.float32), dtype=torch.float32, device=device).unsqueeze(0)
    train_min = torch.tensor(train_min_np, dtype=torch.float32, device=device).unsqueeze(0)
    train_max = torch.tensor(train_max_np, dtype=torch.float32, device=device).unsqueeze(0)

    model.eval()
    for _ in range(n_iters):
        delta = delta.clone().detach().requires_grad_(True)
        x_adv = x_orig + delta
        pred = model(x_adv)
        target_normal = torch.zeros_like(pred)
        loss = criterion(pred, target_normal)

        model.zero_grad(set_to_none=True)
        loss.backward()

        with torch.no_grad():
            step = alpha * delta.grad.sign() * feature_mask
            delta = delta - step
            delta = torch.clamp(delta, min=-eps, max=eps)
            x_adv = x_orig + delta
            x_adv = torch.max(torch.min(x_adv, train_max), train_min)
            delta = x_adv - x_orig
            delta = torch.clamp(delta, min=-eps, max=eps)
            x_adv = x_orig + delta
            x_adv = torch.max(torch.min(x_adv, train_max), train_min)

    return x_adv.detach().cpu().numpy().squeeze(0)


def run_task2_constrained_pgd(
    model: nn.Module,
    device: torch.device,
    X_train: np.ndarray,
    X_test: np.ndarray,
    y_test_bin: np.ndarray,
    y_test_multiclass: np.ndarray,
    y_pred_test: np.ndarray,
    feature_names_after_encoding: list[str],
    scaler: StandardScaler,
    features_json_path: Path,
) -> pd.DataFrame:
    eps_values = [0.05, 0.1, 0.15, 0.2, 0.3]
    n_iters = 40
    alpha = 0.01

    train_min = X_train.min(axis=0)
    train_max = X_train.max(axis=0)

    high_mask, high_or_partial_mask = build_modifiability_masks(feature_names_after_encoding, features_json_path)
    attack_candidates_mask = (y_test_bin == 1) & (y_pred_test == 1)
    if not attack_candidates_mask.any():
        return pd.DataFrame()

    X_candidates = X_test[attack_candidates_mask]
    y_candidates_multiclass = y_test_multiclass[attack_candidates_mask]
    checker = AttackPlausibilityChecker(feature_names_after_encoding)

    def evaluate_mask(eps: float, mask: np.ndarray, mask_name: str) -> dict:
        successful = 0
        plausible_successful = 0

        for i in range(X_candidates.shape[0]):
            x_adv = pgd_evasion_single(
                model=model,
                device=device,
                x_orig_np=X_candidates[i],
                eps=eps,
                alpha=alpha,
                n_iters=n_iters,
                feature_mask_np=mask,
                train_min_np=train_min,
                train_max_np=train_max,
            )

            pred_adv = predict_binary(model, device, x_adv.reshape(1, -1))[0]
            if pred_adv != 0:
                continue

            successful += 1
            attack_family = map_attack_label_to_family(str(y_candidates_multiclass[i]))
            is_plausible, _ = checker.check_plausibility(
                x_adversarial=x_adv,
                attack_type=attack_family,
                scaler=scaler,
                verbose=False,
            )
            if is_plausible:
                plausible_successful += 1

        ratio = (plausible_successful / successful) if successful > 0 else 0.0
        return {
            "Epsilon": eps,
            "Feature Set": mask_name,
            "Successful Evasions": successful,
            "Plausible Attack Ratio": ratio,
        }

    results = []
    for eps in eps_values:
        row = evaluate_mask(eps, high_mask, "HIGHLY_MODIFIABLE")
        if row["Successful Evasions"] == 0:
            row = evaluate_mask(eps, high_or_partial_mask, "HIGHLY_OR_PARTIALLY_MODIFIABLE")
        results.append(row)

    return pd.DataFrame(results)


task2_table = run_task2_constrained_pgd(
    model=model,
    device=device,
    X_train=X_train,
    X_test=X_test,
    y_test_bin=y_test_bin,
    y_test_multiclass=y_test_multiclass,
    y_pred_test=y_pred_test,
    feature_names_after_encoding=feature_names_after_encoding,
    scaler=scaler,
    features_json_path=features_json_path,
)

print("\nTask 2: Constrained PGD results")
if task2_table.empty:
    print("No eligible attack candidates for Task 2.")
else:
    print(task2_table.to_string(index=False, float_format=lambda x: f"{x:.4f}"))


Task 2: Constrained PGD results
 Epsilon       Feature Set  Successful Evasions  Plausible Attack Ratio
  0.0500 HIGHLY_MODIFIABLE                    4                  0.0000
  0.1000 HIGHLY_MODIFIABLE                    7                  0.0000
  0.1500 HIGHLY_MODIFIABLE                   11                  0.0000
  0.2000 HIGHLY_MODIFIABLE                   15                  0.0000
  0.3000 HIGHLY_MODIFIABLE                   21                  0.0000


### Task 3: PGD with Plausibility-Aware Loss

In [5]:
from sklearn.naive_bayes import GaussianNB


def train_gnb_multiclass(X_train: np.ndarray, y_train_multiclass: np.ndarray):
    gnb = GaussianNB()
    gnb.fit(X_train, y_train_multiclass)

    class_to_idx = {c: i for i, c in enumerate(gnb.classes_)}
    params = {
        "means": torch.tensor(gnb.theta_, dtype=torch.float32),
        "vars": torch.tensor(gnb.var_, dtype=torch.float32),
        "class_to_idx": class_to_idx,
    }
    return gnb, params


def gnb_negative_log_likelihood(x_adv: torch.Tensor, class_idx: int, gnb_params) -> torch.Tensor:
    mu = gnb_params["means"][class_idx].to(x_adv.device)
    var = gnb_params["vars"][class_idx].to(x_adv.device)
    var = torch.clamp(var, min=1e-8)

    nll = 0.5 * torch.log(2 * torch.pi * var) + ((x_adv.squeeze(0) - mu) ** 2) / (2 * var)
    return nll.sum()


def pgd_evasion_single_with_plausibility(
    model: nn.Module,
    device: torch.device,
    x_orig_np: np.ndarray,
    original_attack_label: str,
    eps: float,
    alpha: float,
    n_iters: int,
    feature_mask_np: np.ndarray,
    train_min_np: np.ndarray,
    train_max_np: np.ndarray,
    gnb_params,
    lambda_plaus: float = 0.1,
) -> np.ndarray:
    criterion = nn.BCELoss()

    if original_attack_label not in gnb_params["class_to_idx"]:
        return x_orig_np.copy()

    class_idx = gnb_params["class_to_idx"][original_attack_label]

    x_orig = torch.tensor(x_orig_np, dtype=torch.float32, device=device).unsqueeze(0)
    delta = torch.zeros_like(x_orig)

    feature_mask = torch.tensor(feature_mask_np.astype(np.float32), dtype=torch.float32, device=device).unsqueeze(0)
    train_min = torch.tensor(train_min_np, dtype=torch.float32, device=device).unsqueeze(0)
    train_max = torch.tensor(train_max_np, dtype=torch.float32, device=device).unsqueeze(0)

    model.eval()
    for _ in range(n_iters):
        delta = delta.clone().detach().requires_grad_(True)
        x_adv = x_orig + delta
        pred = model(x_adv)

        target_normal = torch.zeros_like(pred)
        l_classifier = criterion(pred, target_normal)
        l_plausibility = gnb_negative_log_likelihood(x_adv, class_idx, gnb_params)
        l_total = l_classifier + lambda_plaus * l_plausibility

        model.zero_grad(set_to_none=True)
        l_total.backward()

        with torch.no_grad():
            step = alpha * delta.grad.sign() * feature_mask
            delta = delta - step
            delta = torch.clamp(delta, min=-eps, max=eps)
            x_adv = x_orig + delta
            x_adv = torch.max(torch.min(x_adv, train_max), train_min)
            delta = x_adv - x_orig
            delta = torch.clamp(delta, min=-eps, max=eps)
            x_adv = x_orig + delta
            x_adv = torch.max(torch.min(x_adv, train_max), train_min)

    return x_adv.detach().cpu().numpy().squeeze(0)


def run_task3_plausibility_aware_pgd(
    model: nn.Module,
    device: torch.device,
    X_train: np.ndarray,
    X_test: np.ndarray,
    y_train_multiclass: np.ndarray,
    y_test_bin: np.ndarray,
    y_test_multiclass: np.ndarray,
    y_pred_test: np.ndarray,
    feature_names_after_encoding: list[str],
    scaler: StandardScaler,
    features_json_path: Path,
) -> pd.DataFrame:
    eps_values = [0.05, 0.1, 0.15, 0.2, 0.3]
    n_iters = 40
    alpha = 0.01
    lambda_plaus = 0.1

    train_min = X_train.min(axis=0)
    train_max = X_train.max(axis=0)

    _, gnb_params = train_gnb_multiclass(X_train, y_train_multiclass)
    high_mask, high_or_partial_mask = build_modifiability_masks(feature_names_after_encoding, features_json_path)

    attack_candidates_mask = (y_test_bin == 1) & (y_pred_test == 1)
    train_attack_labels = set(y_train_multiclass[y_train_multiclass != "normal"])
    non_zero_day_mask = np.array(
        [(lbl in train_attack_labels) and (lbl != "normal") for lbl in y_test_multiclass],
        dtype=bool,
    )
    task3_mask = attack_candidates_mask & non_zero_day_mask
    if not task3_mask.any():
        return pd.DataFrame()

    X_candidates = X_test[task3_mask]
    y_candidates_multiclass = y_test_multiclass[task3_mask]
    checker = AttackPlausibilityChecker(feature_names_after_encoding)

    def evaluate_mask(eps: float, mask: np.ndarray, mask_name: str) -> dict:
        successful = 0
        plausible_successful = 0

        for i in range(X_candidates.shape[0]):
            lbl = str(y_candidates_multiclass[i])
            x_adv = pgd_evasion_single_with_plausibility(
                model=model,
                device=device,
                x_orig_np=X_candidates[i],
                original_attack_label=lbl,
                eps=eps,
                alpha=alpha,
                n_iters=n_iters,
                feature_mask_np=mask,
                train_min_np=train_min,
                train_max_np=train_max,
                gnb_params=gnb_params,
                lambda_plaus=lambda_plaus,
            )

            pred_adv = predict_binary(model, device, x_adv.reshape(1, -1))[0]
            if pred_adv != 0:
                continue

            successful += 1
            attack_family = map_attack_label_to_family(lbl)
            is_plausible, _ = checker.check_plausibility(
                x_adversarial=x_adv,
                attack_type=attack_family,
                scaler=scaler,
                verbose=False,
            )
            if is_plausible:
                plausible_successful += 1

        ratio = (plausible_successful / successful) if successful > 0 else 0.0
        return {
            "Epsilon": eps,
            "Feature Set": mask_name,
            "Successful Evasions": successful,
            "Plausible Attack Ratio": ratio,
        }

    results = []
    for eps in eps_values:
        row = evaluate_mask(eps, high_mask, "HIGHLY_MODIFIABLE")
        if row["Successful Evasions"] == 0:
            row = evaluate_mask(eps, high_or_partial_mask, "HIGHLY_OR_PARTIALLY_MODIFIABLE")
        results.append(row)

    return pd.DataFrame(results)


task3_table = run_task3_plausibility_aware_pgd(
    model=model,
    device=device,
    X_train=X_train,
    X_test=X_test,
    y_train_multiclass=y_train_multiclass,
    y_test_bin=y_test_bin,
    y_test_multiclass=y_test_multiclass,
    y_pred_test=y_pred_test,
    feature_names_after_encoding=feature_names_after_encoding,
    scaler=scaler,
    features_json_path=features_json_path,
)

print("\nTask 3: Plausibility-aware PGD results")
if task3_table.empty:
    print("No eligible non-zero-day attack candidates for Task 3.")
else:
    print(task3_table.to_string(index=False, float_format=lambda x: f"{x:.4f}"))


Task 3: Plausibility-aware PGD results
 Epsilon                    Feature Set  Successful Evasions  Plausible Attack Ratio
  0.0500 HIGHLY_OR_PARTIALLY_MODIFIABLE                   52                  0.1538
  0.1000 HIGHLY_OR_PARTIALLY_MODIFIABLE                  108                  0.1111
  0.1500 HIGHLY_OR_PARTIALLY_MODIFIABLE                  153                  0.1242
  0.2000              HIGHLY_MODIFIABLE                    1                  1.0000
  0.3000              HIGHLY_MODIFIABLE                    1                  1.0000


### Task 4: Analysis Questions

#### Q1 (Task 2 vs Task 3):
From the printed outputs, the average plausible attack ratio increased from 0.0000 (Task 2)
to 0.4778 (Task 3). This is a clear improvement in plausibility.

#### Conclusion:
Based on these observed numbers, Task 3 should be preferred over Task 2 for generating
more realistic adversarial examples, even if some evasion-performance trade-off exists.

#### Q2 (Why this approach is better than alternatives):
Including plausibility directly in the optimization objective (Task 3) guides every update
step toward realistic regions. In contrast, post-hoc filtering only removes bad samples
after generation, and hard rejection during optimization can block useful progress.